## Oppgave 4

In [ ]:
import scipy as sp
import numpy as np
import matplotlib.pyplot as plt
from numba import njit
from tqdm import tqdm

from utilities.probs_fast import p_minus, p_plus
np.seterr(over='ignore')


@njit
def V_1(x : np.array):
    "Sagtannspotensiale"
    
    #Mapper x til periodisiteten
    L = -(1 - alpha) * N_x
    x = L + (x - L) % N_x

    #delt funksjonsuttrykk
    x_less = (0 <= x) & (x <= alpha*N_x)
    x_more = (-(1-alpha)*N_x < x) & (x <= 0)
    x[x_less] = k * x[x_less]/ (alpha*N_x)
    x[x_more] = -k*x[x_more]/((1-alpha)*N_x)
    
    return x

@njit
def V_2(x):
    "Konstant potensiale"
    return np.full_like(x, k, dtype=np.float64)

@njit
def gen_walk_masks(N_particles, beta, V_xm1, V_xp1, V_x0):
    '''Regner ut p minus samt p pluss og bruker uniform distribusjon til
    å bestemme om hver partikkel skal gå venstre, høyre eller ikke bevege seg'''
    uniform_dist = np.random.uniform(0, 1, N_particles)
    go_left = uniform_dist <= (p_minus(beta, V_xm1, V_xp1, V_x0))
    go_right = uniform_dist >= 1-(p_plus(beta, V_xm1, V_xp1, V_x0))
    return go_left, go_right

@njit
def random_walk_ratchet_potential(particles, N_timesteps, T_p, N_particles, N_points, b, beta):
    "Simulerer virrevandring i aksjonspotensiale med frastøtning"
    np.random.seed(120) #Seeder random for å sikre reproduserbare resultater

    #Array for partikkelstrømmer for en syklus slik at snittet kan utregnes
    particle_current_buffer = np.zeros(2*T_p)


    #Definerer under ulike arrays som kan holde på plotteverdier til senere

    #Velger tilfeldige partikler som plottes
    plt_particle_idxs = np.random.choice(np.arange(0, N_particles), size = min(5, N_particles), replace=False)
    particle_movements = np.zeros((N_timesteps, min(5, N_particles)))
    cycle_averaged_particle_currents = np.zeros(N_timesteps // 2*T_p)
    vline_plot_points = vline_plot_points = np.zeros((N_timesteps // T_p, 2), dtype=np.int64) # Brukes i plotting senere for å indikere potensialbytte
    potential_switch_count = 0



    for j in range(N_timesteps):
        n_plus = 0
        n_minus = 0

        #Itererer periodisk gjennom potensialer etter T_p tidssteg
        if (j % T_p == 0) & (j != 0):
            potential_switch_count += 1
            curr_potential = potential_switch_count % 2
            vline_plot_points[j//T_p] = np.array([j, curr_potential])


        x0 = particles
        if curr_potential == 0:
            V_xm1, V_xp1, V_x0 = V_1(x0 -1), V_1(x0 + 1), V_1(x0)
        else:
            V_xm1, V_xp1, V_x0 = V_2(x0 -1), V_2(x0 + 1), V_2(x0)
        go_left, go_right = gen_walk_masks(N_particles, beta, V_xm1, V_xp1, V_x0)

        #Velger tilfeldig rekkefølge på partikkelbevegelse
        movement_order = np.random.permutation(np.arange(N_particles))
        for curr_particle_idx in movement_order:
            distance_from_particle = np.copy(particles) - particles[curr_particle_idx]
            if go_left[curr_particle_idx]:
                if np.any((distance_from_particle > -b) & (distance_from_particle < 0)):
                    continue
                else:
                    #Denne if-statementen sjekker om periodisk randbetingelse er brutt
                    if curr_particle_idx <= b:
                        boundary = N_points - 1 - (b - curr_particle_idx) # -1 pga 0-indeksering
                        if np.any(distance_from_particle >= boundary):
                            continue
                
                #Denne koden kjører kun dersom ingen if-statements ovenfor er gyldig
                particles[curr_particle_idx] -= 1
                n_minus += 1

            elif go_right[curr_particle_idx]:
                if np.any((distance_from_particle < b) & (distance_from_particle > 0)):
                    continue
                if curr_particle_idx >= N_points - 1 - b:
                    boundary = b - (N_points - 1 - curr_particle_idx)
                    if np.any(np.abs(distance_from_particle) >= N_points - 1 - boundary):
                        continue

                particles[curr_particle_idx] += 1
                n_plus += 1

            #Syklisk randbetingelser for x
            if particles[curr_particle_idx] >= N_points:
                particles[curr_particle_idx] = 0
            
            elif particles[curr_particle_idx] <= -1:
                particles[curr_particle_idx] = N_points - 1
        
        
        normalized_particle_current = (n_plus - n_minus) / N_particles
        particle_current_buffer[j % len(particle_current_buffer)] = normalized_particle_current
        particle_movements[j] = (particles[plt_particle_idxs])

        if (j % (2*T_p) == 0) & (j != 0):
            cycle_averaged_particle_currents[N_timesteps // (2*j)] = np.average(particle_current_buffer)

    return potential_switch_count, cycle_averaged_particle_currents, particle_movements, vline_plot_points



class ratchet_interaction_walker():
    "Klasse for å simulere oppgave 4. Hovedsakelig laget for lettere implementering av kode"
    def __init__(self, cfg : dict):
        self.cfg = cfg
        k_b = sp.constants.Boltzmann
        self.potentials = [V_1, V_2]
        self.T = cfg['T']
        self.beta = (self.T * k_b)**-1
        self.beta_k_ratio = cfg['beta_k_ratio']
        self.k = self.beta_k_ratio * (self.T * k_b)
        self.b = cfg['b']
        self.T_p = cfg['T_p']
        self.N_cycles = cfg['N_cycles']
        self.alpha = cfg['alpha']
        self.N_x = cfg['N_x']
        self.N_points = self.N_x * cfg['N_s']
        self.N_particles = cfg['N_p']
        self.N_timesteps = self.T_p * 2 * self.N_cycles


        #Må gjøre noen av variablene globale for at potensial-funksjonene skal kunne brukes
        global k
        global alpha
        global N_x
        k = self.k
        alpha = self.alpha
        N_x = self.N_x
  
    def plot_sawtooth_potential(self):
            "Visualiserer potensiallandskapet i sagtannsfasen"
            x = np.linspace(0,1000, num=10000)
            V_vals = V_1(x)
            plt.plot(x, V_vals)
            plt.show()

    def interaction_simulator(self):
        '''Simulerer "Random walk in a ratchet potential with interactions". Returnerer alle tidssteg samt alle x-posisjoner til tilfeldig utvalgte partikler'''

        if self.N_particles == 1:
            particles = np.array([self.N_points // 2])
        else:
            particles = np.linspace(0, self.N_points -1, self.N_particles, dtype=np.int64)

        self.potential_switch_count, self.cycle_averaged_particle_currents, particle_movements, self.vline_plot_points = random_walk_ratchet_potential(
            particles,
            self.N_timesteps,
            self.T_p,
            self.N_particles,
            self.N_points,
            self.b,
            self.beta
        )

        particle_movements = np.array(particle_movements).T
        #Tar gjennomsnittet av alle syklus-snittede partikkelstrømningene for å få ett enkelt tall
        self.cycle_averaged_particle_currents = np.average(self.cycle_averaged_particle_currents)


        return (np.arange(self.N_timesteps), particle_movements)
    


def plot_particle_movement(walker : ratchet_interaction_walker, cfg : dict, x_array : np.array, T : np.array, oppg : str, plot_potential_switches = True):
    "PLotter posisjonen til et forhåndsvalgt antall partikler gjennom simuleringen"
    plt.figure(2)
    cfg = cfg[oppg]
    for x in x_array:

        #Fjerner diskontinuiteter fra plottet [KI generert]
        x_float = x.astype(np.float64)
        diffs = np.abs(np.diff(x_float)) 
        jump_indices = np.where(diffs >= (cfg['N_x']*cfg['N_s'] - 10))[0]
        for idx in jump_indices:
            if idx + 1 < len(x_float):
                x_float[idx + 1] = np.nan



        plt.plot(T, x_float)
    
    #Plotter vertikale linjer som viser potensialbytte om ønskelig
    if plot_potential_switches:
        for t, potential_idx in walker.vline_plot_points:
            if potential_idx % 2 == 0:
                color = 'green'
                label = '$V_1$' if t == walker.vline_plot_points[0][0] else ''
            else:
                color = 'black'
                label = '$V_2$' if t == walker.vline_plot_points[1][0] else ''
            plt.axvline(t, color=color, linestyle='--', label = label)
    else:
        plt.grid()
    plt.legend()
    plt.title('Partikkelbevegelse over tid i aksjonspotensiale med frastøtning')
    plt.xlabel('Tidssteg')
    plt.ylabel('Plassering')
    plt.show()

def rho_iterator(cfg: dict, oppg: str, T_p = 'default'):
    '''Finner alle syklus-snittede partikkelstrømninger for oppgitt intervall av rho. 
    Returnerer:         \n
    -Walker-objektet    \n
    -Liste med tuppler av (rho-verdien, liste over alle cycle-averaged particle currents for rho-verdien)    \n
    -Liste med utvalg av partikkelbevegelser for gitt rho
    '''

    rho_current_values = list()
    x_t_values = list()
    cfg = cfg[oppg]

    #Endrer T_p dersom ønskelig, men holder T_p * N_c = 30 000
    if not isinstance(T_p, str):
        cfg['T_p'] = int(T_p)
        cfg['N_cycles'] = int(3E4 / T_p)

    #Finner antall partikler som tilsvarer oppgitt intervall for partikkeltettheter i oppgaven
    N_p_min, N_p_max = np.ceil(np.array([cfg['rho_min'],cfg['rho_max']]) * cfg['N_s'] * cfg['N_x'] / cfg['b']).astype(int)
    for N_p in tqdm(np.arange(N_p_min, N_p_max + 1)):
        cfg['N_p'] = N_p
        rho = N_p * cfg['b'] / (cfg['N_x'] * cfg['N_s'])
        walker = ratchet_interaction_walker(cfg)
        T, x_array = walker.interaction_simulator()
        x_t_values.append([T, x_array])
        rho_current_values.append([rho, walker.cycle_averaged_particle_currents])
        

    rho_current_values = np.array(rho_current_values).T
    return walker, rho_current_values, x_t_values



#### a)

In [ ]:


def oppg4a(cfg : dict):
    walker = ratchet_interaction_walker(cfg['4a'])
    T, x_array = walker.interaction_simulator()

    plot_particle_movement(walker, cfg, x_array, T, '4a')

cfg = {"alpha": 0.2,
  "T": 310.15,
  "N_x": 20,
  "T_p": 40,
  "N_cycles": 5,
  "N_s": 4,
  "h": 1,
  "beta_k_ratio": 1000,
  "N_p": 5,
  "b": 2}
  
oppg4a(cfg)

Som plottet viser har alle partiklene to ulike typer bevegelse i potensiallandskapet. Ved konstant potensiale $V(x) = k$ er alle sannsynlighetene for bevegelse like, så partiklene diffunderer. Ved bytte til sagtannspotensialet beveger deretter alle partiklene seg i retning av nærmeste potensialbrønn og forblir der. Forskjellen mellom oppførselen her og i oppgave 3 er naturligvis at ettersom vi modellerer avstøtning mellom partiklene kan de ikke oppta samme punkt. Dermed vil kun en partikkel kunne nå bunnen av en gitt potensialbrønn, og alle andre partikler som faller mot samme brønn vil forbli liggende "oppå" denne partikkelen i potensialfeltet. I denne modellen vil dette bety at hver påfølgende partikkel som legges til i en potensialbrønn allerede inntatt av en annen partikkel vil ha høyere sannsynlighet for å bevege seg videre i potensialet til en annen brønn, og dermed skape partikkelstrømning selv i sagtannspotensialet. Dette er jo da gitt at sagtannspotensialet er usymmetrisk, altså $\alpha\neq0.5$.

#### b)

In [ ]:



cfg = {
  "alpha": 0.2,
  "T": 310.15,
  "N_x": 100,
  "T_p": 300,
  "N_cycles": 100,
  "N_s": 10,
  "h": 1,
  "beta_k_ratio": 1000,
  "N_p": 100,
  "b": 20, # Partikkelstørrelse
  "rho_min": 0.01,
  "rho_max": 1}


def oppg4b(cfg : dict):
    plot_values = rho_iterator(cfg, '4b')
    walker, rho_current_values, x_t_values = plot_values
    rho, avg_current = rho_current_values

    plt.plot(rho, avg_current)
    plt.title('Syklus-snittet partikkelstrømning med varierende partikkeltetthet')
    plt.xlabel('$\\rho$')
    plt.ylabel('Normalisert partikkelstrøm')
    plt.show()


oppg4b(cfg)

Økende partikkeltetthet holder partikkelstrømningen relativt konstant inntil omtrent $\rho = 0.6$, og deretter synker strømningen jevnlig. 

#### c)

In [ ]:
cfg = {
  "alpha": 0.2,
  "T": 310.15,
  "N_x": 100,
  "T_p": 300,
  "N_cycles": 100,
  "N_s": 10,
  "h": 1,
  "beta_k_ratio": 1000,
  "N_p": 100,
  "b": 20, # Partikkelstørrelse
  "rho_min": 0.01,
  "rho_max": 1}


def oppg4c(cfg : dict):
    T_p_vals = np.array([100, 300, 600, 1000])
    for T_p in T_p_vals:
        print(f'---------------- T_P = {T_p} ----------------')
        plot_vals = rho_iterator(cfg, '4b', T_p)
        walker, rho_current_values, x_t_values = plot_vals
        rho, avg_current = rho_current_values

        plt.plot(rho, (avg_current), label=f'$T_p = {T_p}$')
        plt.title('Syklus-snittet partikkelstrømning med varierende partikkeltetthet')
        plt.xlabel('$\\rho$')
        plt.ylabel('Normalisert partikkelstrøm')
        plt.legend()
        plt.show()
        plt.clf()

oppg4c(cfg)

Plottene ovenfor viser at strømningsfunksjonens maksimum skjer for mindre og mindre partikkeltetthet med voksende antall tidssteg. Det er hovedsakelig to ulike fenomener som forklarer denne oppførselen. For lave nok $T_p$ vil ikke partiklene rekke å diffundere fullstendig mellom byttene av potensiale. Dette betyr at for lave tettheter vil alle partiklene være bundet til potensialbrønnene, og de får kun byttet dersom de under diffunderingsfasen tilfeldigvis beveger seg svært rettet mot en annen brønn. Med økende tetthet vil etterhvert flere partikler ligge i samme brønn. På grunn av det usymmetriske potensialet vil frastøtningen føre til at partiklene som er høyere opp i potensialfeltet posisjonsmessig ligger nærmere neste brønn, slik at de har høyere sannsynlighet for å bevege seg videre under konstant potensiale. Totalt er det dermed flere muligheter for en partikkel å bytte brønn, noe som fører til økt netto strømning. Når man derimot når en kritisk stor $\rho$ vil den positive effekten dette gir bli overskygget av blokkeringen som skjer ved for mange nabopartikler, og dermed minker netto strømning igjen.

Når $T_p$ blir tilstrekkelig stor vil partiklene så og si alltid ha tid til å diffundere til uniform fordeling uansett partikkeltetthet. Dette betyr at selv en enkelt partikkel vil kunne bytte brønn hver syklus. Dermed vil det å øke partikkeltettheten ikke ha noen positiv innvirkning i strømningen til hver enkelt partikkel. Den eneste mulige effekten er negativ, ved at to partikler møter hverandre og dermed potensielt hindrer hverandres bevegelse til neste brønn. Konsekvensen av dette for strømingene er at maksimum verdi skjer ved $N_p = 1$, og at alle påfølgende verdier vil være mindre eller lik denne strømmen. Dette er da fordi vi spesifikt undersøker normalisert partikkelstrøm.

Det er også verdt å merke at størrelsesordenen på partikkelstrømningen minker med økende $T_p$. Dette kommer av måten vi regner ut den syklus-snittede partikkelstrømmen på. Ved konstant potensiale har vi fra tidligere oppgaver forklart at netto strømning er rundt null. All strømningen i sagtannspotensialet skjer ved et omtrent konstant antall tidssteg ut ifra partikkeltettheten. Dette betyr at ettersom syklus-snittet strømning regnes ut ved å dele på antall tidssteg men selve strømningen skjer ved konstant tid blir resultatet at den gjennomsnittlige strømningen blir mindre.

#### d)
Ettersom det var utfordrende å finne en artikkel som oppfyllte alle spesifikasjonene optimalt, ble en artikkel valgt som ikke tar for seg "hard core repulsion", men heller Coulumb-krefter. Hver partikkel har derfor alle ladning lik -e, og det er bare disse kreftene som virker mellom partikklene. Artikkelen var fremdeles relevant for proskjektet vårt, og eksakt hvor ulikt det hadde vært med en hardkjærne-approksimasjon er uvisst. Partikklene vil nok spre seg gjevnere med Coulumb-krefter, men det er trolig mye likt.

Videre modellerer de et pulserende potensialet, slik som vi. Men de gjør dette i 2D og bruker et asymmetrisk potensial som er formet som en sinus-bølge og går gradvis fra negativt til positivt potensialet og tilbake. Partikklene som beveger seg måles fremdeles hvor langt de går i x-retning, men de har altså rom å bevege seg i y-retning. De fokuserer på å se på ulike partikkeltettheter og pulseringsfrekvenser.

Den finner at ved høyere partikkeltetthet kan man øke frekvensen til det pulserende potensialet, og man vil derfra få en større partikkelstrøm. Dette siden flere av partikklene kolliderer noe som fører til at mange av partikklene vil befinne seg nærmere neste intervall, og trenger kortere tid for å diffundere. De er på sett og vis allerede påbegynt diffusjoenen. Dette ser man på figuren. De fargede tallene er antallet partikkler i et gitt system. Ved ikke-interagerende partikkler vil man derfor ikke få denne effekten og alle partikklene vil fortere sette seg "fast" i sitt intervall ved en høyere frekvens, da alle vil dra fra punktet med minst potensialet og rekker ikke over til neste intervall før frekvensen er tilbake til start.

Det andre de fant var at ved høy partikkeltetthet og lav frekvens får noen partikkler så mye energi fra å kollidere at de hopper over flere intervaller. Dette skjedde ikke for høye frekvenser.

![alt text](image-1.png)

https://pubs.acs.org/doi/pdf/10.1021/acs.jpcc.9b00344?ref=article_openPDF
